# Polarization Switching Barrier of Ferroelectric HfO₂ along the S:T Pathway

> **Xinpeng Mu, Yao Wu, Binjian Zeng, Jie Jiang, Yichun Zhou, Lu Yin, Min Liao, and Qiong Yang**
> "Polarization switching of HfO2 ferroelectric in bulk and electrode/ferroelectric/electrode heterostructure"
> npj Computational Materials 11, 126 (2025).
> [DOI: 10.1038/s41524-025-01633-2](https://doi.org/10.1038/s41524-025-01633-2)

Computes the barrier of the S:T polarization reversal of orthorhombic Pca2₁ HfO₂ from the two materials created in the
[structure notebook](perturbation_ferroelectric_switching_hafnium_dioxide.ipynb): the initial structure, Fig. 1(a), and
the switched one, Fig. 1(c), are relaxed with PBE on the platform with the cell fixed, and a climbing-image nudged
elastic band (NEB) runs between them. The barrier is compared with the manuscript's Table 1, 0.736 eV per unit cell
(four Hf and eight O) for bulk HfO₂ with the cell fixed, a value it takes from Wu et al. (2023).

- The manuscript's own calculations use VASP with PAW and PBE at 500 eV and a 6 × 6 × 6 Monkhorst–Pack grid; the settings behind the Table 1 value are those of Wu et al. (2023).
- HfO₂ is at Standata's lattice, 5.2195 × 5.0048 × 5.0327 Å (manuscript: 5.256 × 5.046 × 5.073 Å).

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](perturbation_ferroelectric_switching_hafnium_dioxide.ipynb), which saves `HfO2 Pca21` and `HfO2 Pca21 S:T switched` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the calculations to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the initial and the switched structure by name and print their atom counts.
1. Configure the shared DFT model: the DFT model and the k-grid of the relaxations and the NEB.
1. Configure compute: create a compute configuration on the cluster of `CLUSTER_NAME`, or on the first listed cluster if it is not set.
1. Relax both structures on the platform with the cell fixed.
1. Run one climbing-image NEB between the relaxed structures.
1. Retrieve the results: the energy profile, the barrier and the largest image error.
1. Compare with Mu et al. (2025), Table 1.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Materials and the manuscript's value

In [ ]:
# Names saved by perturbation_ferroelectric_switching_hafnium_dioxide.ipynb: the first and the last image of the path
INITIAL_MATERIAL_NAME = "HfO2 Pca21"
FINAL_MATERIAL_NAME = "HfO2 Pca21 S:T switched"
PATH_NAME = "S:T"
# Mu et al. 2025, Table 1, bulk, cell fixed (from Wu et al. 2023): the barrier in eV per unit cell (Hf4O8)
MANUSCRIPT_BARRIER = 0.736

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
NEB_WORKFLOW_SEARCH_TERM = "neb.json"
APPLICATION_NAME = "espresso"
APPLICATION_VERSION = "7.5"

# full host name of the cluster
CLUSTER_NAME = None
# 36 cores fit the OF node of every cluster
QUEUE_NAME = QueueName.OF
NODES = 1
PPN = 36
TIME_LIMIT_HOURS = 4
TIME_LIMIT = f"{TIME_LIMIT_HOURS:02d}:00:00"
# pw.x and neb.x stop 10 minutes before TIME_LIMIT and write their files
MAX_SECONDS = TIME_LIMIT_HOURS * 3600 - 600

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
# manuscript: PAW at 500 eV; ours GBRV ultrasoft pseudopotentials at their converged 40/200 Ry
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry
ECUTRHO = 200  # Ry
# the manuscript's 6x6x6 Monkhorst-Pack grid: for an even grid it is offset from Gamma by half a step (shift 1)
KGRID = [6, 6, 6]
KGRID_SHIFTS = [1, 1, 1]
# &SYSTEM of the relaxations and the NEB: without spin and with 0.005 Ry smearing (the NEB workflow defaults to
# nspin = 2 and 0.03 Ry); the manuscript states neither
QE_SYSTEM_SETTINGS = {"nspin": 1, "degauss": 0.005}
# &CONTROL of the relaxations: the cell fixed, until every force component is below QE's default 1e-3 Ry/Bohr
RELAXATION_SETTINGS = {"forc_conv_thr": 1e-3, "max_seconds": MAX_SECONDS}
# images between the end points; the path is converged when the largest image error is below 0.1 eV/Å
N_IMAGES = 1
# neb.x stops after nstep_path steps, converged or not; the largest image error is printed in 9
NEB_PATH_SETTINGS = {"nstep_path": 50, "minimum_image": True}

RELAX_UNIT = "pw_relax"
NEB_UNIT = "neb"
# written by neb.x: reaction coordinate, energy and error of every image of the final path
NEB_ENERGY_FILE = "__prefix__.dat"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the initial and the switched structure from the uploads folder or the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material

material_names = [INITIAL_MATERIAL_NAME, FINAL_MATERIAL_NAME]
materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in material_names}
for name, material in materials_by_name.items():
    print(f"{name}: {material.basis.number_of_atoms} atoms")

## 4. Configure the shared DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory
from mat3ra.wode.context.providers import PointsGridDataProvider

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
kgrid_context = PointsGridDataProvider(
    material=materials_by_name[INITIAL_MATERIAL_NAME], dimensions=KGRID, shifts=KGRID_SHIFTS, isEdited=True
).get_context_item_data()
print(f"Using application: {app.name}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
if not CLUSTER_NAME:
    clusters = client.clusters.list()
    print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

cluster = {"hostname": CLUSTER_NAME, "fqdn": CLUSTER_NAME} if CLUSTER_NAME else clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, nodes=NODES, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {NODES}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relaxation of the initial and the switched structure

Each material, as built in the structure notebook, is relaxed with the cell fixed and the settings of 1.4 on the platform; the relaxed structures are the end points of the NEB. A relaxed structure already on the account, from a finished relaxation of the same material with the same k-grid, is reused, as is a relaxation job of the same material that is still queued or running.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_or_create_material

relaxation_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    RELAX_WORKFLOW_SEARCH_TERM
)
relaxed_materials = {}
relax_job_ids = {}
new_job_ids = []
for name, material in materials_by_name.items():
    saved_material = get_or_create_material(client, material, ACCOUNT_ID)
    relaxed_materials[name] = find_relaxed_material(client, material, ACCOUNT_ID, kgrid=KGRID, unit_name=RELAX_UNIT)
    if relaxed_materials[name] is not None:
        continue
    workflow = Workflow.create(relaxation_workflow_config)
    workflow.name = f"Fixed-cell Relaxation {name}"
    for subworkflow in workflow.subworkflows:
        for application in (subworkflow.application, *(unit.application for unit in subworkflow.units)):
            application.version = APPLICATION_VERSION
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
    workflow.subworkflows[0].get_unit_by_name(name=RELAX_UNIT).add_context(kgrid_context)
    patch_workflow_qe_input(workflow, {"control": RELAXATION_SETTINGS, "system": QE_SYSTEM_SETTINGS}, [RELAX_UNIT])
    job = find_job_for_material(client, saved_material["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=KGRID, unit_name=RELAX_UNIT)
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
    relax_job_ids[name] = job["_id"]

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.core.entity.material.api import get_final_structure_for_job

submit_jobs(client.jobs, new_job_ids)
if relax_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, list(relax_job_ids.values()), poll_interval=POLL_INTERVAL)
for name, job_id in relax_job_ids.items():
    relaxed_materials[name] = get_final_structure_for_job(client, job_id)

## 7. Nudged elastic band

The relaxed initial structure is the first image and the relaxed switched structure the last; neb.x interpolates `N_IMAGES` images between them and climbs the highest one to the saddle point. Both list their atoms in the same order, so each atom moves to its own position; with `minimum_image`, an atom that the platform stores wrapped into the cell moves the short way. An NEB job that is queued, running or finished from the same first image with the same k-grid is reused, also when the application version or a setting in 1.4 has changed.

In [ ]:
neb_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    NEB_WORKFLOW_SEARCH_TERM
)
path_materials = [relaxed_materials[INITIAL_MATERIAL_NAME], relaxed_materials[FINAL_MATERIAL_NAME]]
workflow = Workflow.create(neb_workflow_config)
workflow.name = f"Nudged Elastic Band {INITIAL_MATERIAL_NAME} {PATH_NAME}"
for subworkflow in workflow.subworkflows:
    for application in (subworkflow.application, *(unit.application for unit in subworkflow.units)):
        application.version = APPLICATION_VERSION
workflow.subworkflows[0].model = model
apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=NEB_UNIT)
# nowf: neb.x writes no wavefunction files; each image's SCF starts from atomic wavefunctions
patch_workflow_qe_input(workflow, {"control": {"max_seconds": MAX_SECONDS, "disk_io": "nowf"},
                                   "system": QE_SYSTEM_SETTINGS, "path": NEB_PATH_SETTINGS}, [NEB_UNIT])
neb_unit = workflow.subworkflows[0].get_unit_by_name(name=NEB_UNIT)
neb_unit.add_context(kgrid_context)
neb_unit.add_context({"name": "neb", "isEdited": True, "data": {"nImages": N_IMAGES}, "extraData": {}})
job = find_job_for_material(client, path_materials[0].id, workflow.name, ACCOUNT_ID,
                            statuses=("submitted", "queued", "active", "finished"),
                            kgrid=KGRID, unit_name=NEB_UNIT)
if job is None:
    materials_set_name = f"{workflow.name} {path_materials[0].id}-{path_materials[1].id}"
    materials_set = client.materials.create_set({"name": materials_set_name, "owner": {"_id": ACCOUNT_ID}})
    for material in path_materials:
        client.materials.move_to_set(material.id, "", materials_set["_id"])
    job = create_job(
        api_client=client, materials=path_materials, workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        materials_set=materials_set,
    )
    submit_jobs(client.jobs, [job["_id"]])
    print(f"✅ Created NEB job {job['_id']}")
else:
    print(f"♻️  Reusing existing NEB job {job['_id']}")
neb_job_id = job["_id"]

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [neb_job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. Energy profile, barrier and the largest image error

neb.x writes the energy of each image relative to the first one, the initial structure, against the reaction coordinate from 0 to 1; the barrier is the highest.

In [ ]:
from mat3ra.notebooks_utils.io import read_from_url

files = client.jobs.list_files(neb_job_id)
energy_file = next(file for file in files if file["key"].endswith(NEB_ENERGY_FILE))
energy_table = await read_from_url(energy_file["signedUrl"])
reaction_coordinates = [float(line.split()[0]) for line in energy_table.splitlines()]
energies = [float(line.split()[1]) for line in energy_table.splitlines()]
largest_image_error = max(float(line.split()[2]) for line in energy_table.splitlines()[1:-1])
barrier = max(energies)

## 9. Comparison with Mu et al. (2025), Table 1

In [ ]:
print(f"{PATH_NAME} barrier, Pca2₁ HfO₂: ours {barrier:.3f} eV/u.c. | Mu et al. 2025, Table 1 (from Wu et al. 2023), "
      f"{MANUSCRIPT_BARRIER} eV/u.c.")
print(f"Largest image error, {PATH_NAME} path: {largest_image_error:.3f} eV/Å")

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

figure, axes = plt.subplots()
axes.plot(reaction_coordinates, energies, marker="o", label=f"ours, barrier {barrier:.3f}")
axes.plot(reaction_coordinates[energies.index(barrier)], MANUSCRIPT_BARRIER, marker="s", linestyle="none",
          color="gray", label=f"Mu et al. 2025, Table 1, barrier {MANUSCRIPT_BARRIER}")
axes.set_xlabel("Reaction coordinate")
axes.set_ylabel("E − E(initial) (eV/u.c.)")
axes.set_title(f"{PATH_NAME} polarization reversal, Pca2₁ HfO₂")
axes.legend()
display_matplotlib_figure(figure)

## References

[1] X. Mu, Y. Wu, B. Zeng, J. Jiang, Y. Zhou, L. Yin, M. Liao, and Q. Yang, "Polarization switching of HfO2 ferroelectric in bulk and electrode/ferroelectric/electrode heterostructure", npj Comput. Mater. 11, 126 (2025). [DOI: 10.1038/s41524-025-01633-2](https://doi.org/10.1038/s41524-025-01633-2)

[2] Y. Wu, Y. Zhang, J. Jiang, L. Jiang, M. Tang, Y. Zhou, M. Liao, Q. Yang, and E. Y. Tsymbal, "Unconventional polarization-switching mechanism in (Hf,Zr)O2 ferroelectrics and its implications", Phys. Rev. Lett. 131, 226802 (2023). [DOI: 10.1103/PhysRevLett.131.226802](https://doi.org/10.1103/PhysRevLett.131.226802)